# US-003：多格式 EEG 数据导入与导出

**目标：** 掌握常见 EEG 文件格式的读写方法，能处理不同设备和来源的数据。

**涵盖格式：** EDF/BDF、BrainVision、EEGLAB .set、FIF

## 1. MNE 导入函数速查

| 格式 | 来源 | 读取函数 |
|------|------|----------|
| `.fif` | MNE / Neuromag | `mne.io.read_raw_fif()` |
| `.edf` | 医院常用 | `mne.io.read_raw_edf()` |
| `.bdf` | BioSemi | `mne.io.read_raw_bdf()` |
| `.vhdr` | BrainProducts | `mne.io.read_raw_brainvision()` |
| `.set` | EEGLAB | `mne.io.read_raw_eeglab()` |
| `.cnt` | Neuroscan | `mne.io.read_raw_cnt()` |
| `.gdf` | 通用格式 | `mne.io.read_raw_gdf()` |

核心用法一致：`raw = mne.io.read_raw_xxx(filename, preload=False)`

## 2. 读取 FIF 格式（MNE 原生）

FIF 是 MNE 的原生格式，信息保存最完整，推荐作为中间存储格式。

In [ ]:
import mne

# 使用 sample 数据中的 .fif 文件
sample_dir = mne.datasets.sample.data_path()
fif_path = sample_dir / "MEG" / "sample" / "sample_audvis_raw.fif"

raw_fif = mne.io.read_raw_fif(fif_path, preload=True)
print(f"FIF: {raw_fif}")
print(f"通道类型: {set(mne.io.pick.channel_type(raw_fif.info, i) for i in range(len(raw_fif.ch_names)))}")

## 3. 读取 EDF 格式

EDF (European Data Format) 是医院 EEG 最常见的格式。MNE 对 EDF 支持很好。

> 如果本地有 .edf 文件，修改路径即可测试。

In [ ]:
# ===== 示例：读取 EDF 文件 =====
# edf_path = "path/to/your/file.edf"
# raw_edf = mne.io.read_raw_edf(edf_path, preload=True)
# print(raw_edf)
# print(f"通道名: {raw_edf.ch_names}")

# ===== 常见问题：通道名不规范 =====
# EDF 文件的通道名可能是 "EEG Fpz-Cz" 或 "Fpz-Cz" 这种格式
# MNE 可能无法自动识别为 'eeg' 类型，需要手动设置：
# raw_edf.set_channel_types({ch: 'eeg' for ch in raw_edf.ch_names if 'EEG' in ch})

print("EDF 读取示例：请将实际 .edf 文件路径替换到代码中运行。")

## 4. 读取 BrainVision 格式

BrainVision 是 BrainProducts 设备的标准格式，由三个文件组成：
- `.vhdr` — 头文件（主文件，读取时指定它）
- `.vmrk` — 标记文件
- `.eeg` — 二进制数据文件

In [ ]:
# ===== 示例：读取 BrainVision 文件 =====
# vhdr_path = "path/to/your/file.vhdr"
# raw_bv = mne.io.read_raw_brainvision(vhdr_path, preload=True)
# print(raw_bv)

print("BrainVision 读取示例：指定 .vhdr 文件路径即可。")

## 5. 读取 EEGLAB .set 文件

EEGLAB 是 MATLAB 的 EEG 分析工具箱，`.set` + `.fdt` 是其标准格式。
MNE 可以读取 `.set` 文件（需要 `.fdt` 在同一目录下）。

In [ ]:
# ===== 示例：读取 EEGLAB .set 文件 =====
# set_path = "path/to/your/file.set"
# raw_set = mne.io.read_raw_eeglab(set_path, preload=True)
# print(raw_set)

print("EEGLAB .set 读取示例：MNE 会自动找到同目录的 .fdt 文件。")

## 6. 导出为 FIF 格式

MNE 推荐：不管你原始数据什么格式，预处理前先存一份 .fif。
FIF 保存了完整的 Info 和 Annotations，后续加载最快。

In [ ]:
# 用 US-002 的模拟数据演示导出
import numpy as np

# 创建模拟 Raw
n_channels, sfreq, duration = 8, 250, 60
n_times = int(sfreq * duration)
data = np.random.randn(n_channels, n_times) * 20e-6
ch_names = [f'EEG {i:03d}' for i in range(1, n_channels + 1)]
info = mne.create_info(ch_names, sfreq=sfreq, ch_types='eeg')
raw = mne.io.RawArray(data, info)

# 添加一些 Annotations
onsets = [10.0, 25.0, 40.0]
durations = [0.0, 0.0, 0.0]
descriptions = ['stim_1', 'stim_2', 'stim_1']
annot = mne.Annotations(onsets, durations, descriptions)
raw.set_annotations(annot)

# 保存为 FIF
output_path = "dummy_raw.fif"
raw.save(output_path, overwrite=True)
print(f"已保存: {output_path}")

# 验证：读回来
raw_reloaded = mne.io.read_raw_fif(output_path, preload=True)
print(f"重新加载: {raw_reloaded}")
print(f"Annotations 完好: {raw_reloaded.annotations}")

## 7. 格式选择建议

```
原始数据 (.edf / .vhdr / .set)
    │
    │  read_raw_xxx()
    ▼
┌─────────────────────────┐
│  MNE Raw 对象            │
│  ↓ 预处理（滤波、ICA等）  │
│  ↓ raw.save('cleaned.fif')│
│  中间存储：.fif           │
│  ↓ Epoching              │
│  epochs.save('epo.fif')  │
│  分析就绪                 │
└─────────────────────────┘
```

**原则：** 读入后立即存一份 .fif，后续所有操作都从 .fif 加载。

## 8. 练习

1. 如果你手头有 .edf 文件，尝试读取并打印通道名
2. 对比 `preload=True` 和 `preload=False` 时 `raw.get_data()` 的速度
3. 尝试 `raw.crop(tmin=10, tmax=50)` 截取一段，再保存为 .fif